# FMCG Demand Forecasting & Inventory Replenishment

This notebook prepares daily product-level sales, estimates each product's total demand over the next seven days, evaluates time-aware model routes, and creates a preliminary replenishment quantity. It is a research prototype, not a live inventory system.

**Input:** an authorized `historical_data.csv` in the repository root. The file is intentionally excluded from Git. See [the methodology note](../docs/methodology.md) for implementation details.

## 1. Setup and local input

The first cell checks the input path and required columns, then converts the fields used by the existing workflow. It does not embed a dataset or save data into the notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("historical_data.csv")
REQUIRED_COLUMNS = {
    "product_id", "category", "hcode", "sold", "date", "price", "is_promotion"
}
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Place an authorized historical_data.csv in the repository root; the dataset is not included."
    )

df = pd.read_csv(DATA_PATH)
missing_columns = REQUIRED_COLUMNS.difference(df.columns)
if missing_columns:
    raise ValueError(f"Input is missing required columns: {sorted(missing_columns)}")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
invalid_date_count = int(df["date"].isna().sum())
if invalid_date_count:
    raise ValueError(f"Input contains {invalid_date_count} rows with an invalid or missing date.")
df["sold"] = pd.to_numeric(df["sold"], errors="coerce").astype("float32")
df["price"] = pd.to_numeric(df["price"], errors="coerce").astype("float32")
df["is_promotion"] = (
    pd.to_numeric(df["is_promotion"], errors="coerce").fillna(0).astype("int8")
)
df["product_id"] = df["product_id"].astype("category")


## 2. Data quality checks and exploratory analysis

Review schema, missingness, duplicate rows, date coverage, sales and price distributions, promotion patterns, and category/product counts. These cells are intended for local analysis; clear all displayed outputs before committing.


In [ ]:
df.dtypes.to_frame("dtype")

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()


In [ ]:
print(df['category'].value_counts().head(20))


In [ ]:
print("Products:", df['product_id'].nunique())
print("Categories:", df['category'].nunique())
print("H-Codes:", df['hcode'].nunique())

In [ ]:
df['date']=pd.to_datetime(df['date'])

In [ ]:
print(df['date'].max())
print(df['date'].min())

In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week']=df['date'].dt.dayofweek

In [ ]:
df.dtypes.to_frame("dtype")

In [ ]:
df['category'] = df['category'].astype('string').fillna('Unknown').astype('category')


In [ ]:
df.isnull().sum()

### Exploratory sales distributions and trends

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,5))
plt.hist(df['sold'], bins=50)
plt.title('Distribution of Units Sold')
plt.xlabel('Units Sold')
plt.ylabel('Frequency')
plt.show()


In [ ]:
df['sold'].describe()

In [ ]:
plt.figure(figsize=(10,6))
plt.boxplot(df['sold'], vert=False)
plt.title('Sales Outliers')
plt.xlabel('Units Sold')
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
plt.hist(np.log1p(df['sold']), bins=50)
plt.title('Sales Distribution (log scale)')
plt.xlabel('log(1 + sold)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(df['price'], bins=50)
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()


In [ ]:
daily_sales=df.groupby('date')['sold'].sum()

daily_sales.plot()
plt.title('Daily sales Trend')
plt.show()

In [ ]:
monthly_sales=df.groupby(df['date'].dt.to_period('M'))['sold'].sum()

plt.figure(figsize=(15,5))
monthly_sales.plot()
plt.title("Monthly Sales Trend")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
df.boxplot(column='sold', by='is_promotion')
plt.title('Promotion vs Sales')
plt.suptitle('')
plt.xlabel('is_promotion')
plt.ylabel('sold')
plt.show()


In [ ]:
df.groupby('category', observed=True)['sold'].sum().sort_values(ascending=False)

## 3. Daily product-level demand panel

Sales are summed by product and date. The complete product-date grid is reindexed with zero for dates absent from the extract. This encodes an assumption: an absent row means zero observed sales, which may not hold during stockouts, assortment changes, or incomplete reporting.

In [ ]:
full = pd.date_range(df['date'].min(), df['date'].max())

idx = pd.MultiIndex.from_product(
    [df['product_id'].unique(), full],
    names=['product_id', 'date']
)

daily = (
    df.groupby(['product_id', 'date'], observed=True)['sold']
      .sum()
      .reindex(idx, fill_value=0)
      .reset_index()
)

product_category = df[['product_id', 'category']].drop_duplicates('product_id')
daily = daily.merge(
    product_category,
    on='product_id',
    how='left'
)

daily = daily.sort_values(['date', 'product_id']).reset_index(drop=True)

print('daily category count:', daily['category'].nunique())
print('missing category values:', daily['category'].isna().sum())
daily.head()


In [ ]:
df = df.sort_values(['product_id', 'date'])

info = (
    df.groupby(['product_id', 'date'], observed=True)[['price', 'is_promotion', 'hcode']]
      .last()
      .reset_index()
)

daily = daily.drop(
    columns=['price', 'is_promotion', 'hcode'],
    errors='ignore'
)

daily = daily.merge(
    info,
    on=['product_id', 'date'],
    how='left'
)

daily['price'] = (
    daily.groupby('product_id', observed=True)['price']
         .transform(lambda x: x.ffill())
).astype('float32')

daily['is_promotion'] = daily['is_promotion'].fillna(0).astype('int8')

daily['hcode'] = daily['hcode'].astype('string')
daily['hcode'] = (
    daily.groupby('product_id', observed=True)['hcode']
         .transform(lambda x: x.ffill())
)
daily['hcode'] = daily['hcode'].astype('string').fillna('Unknown').astype('category')

daily['product_id'] = daily['product_id'].astype('category')
daily['category'] = daily['category'].astype('category')

daily.head()


### Review sales outliers

In [ ]:
df[['sold']].describe()

In [ ]:
df['sold'].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

In [ ]:
df.sort_values('sold',ascending=False)[['product_id', 'date', 'sold', 'price', 'is_promotion']].head(20)

In [ ]:
df[df['date'] == '2023-08-11'][['product_id', 'sold', 'price', 'is_promotion']].head(20)

In [ ]:
df[df['product_id'] == 'BC5'][['date', 'sold']].sort_values('date')

In [ ]:
df[df['product_id'] == 'BC5']['sold'].describe()

In [ ]:
df[df['product_id']=='BC5'][['date','sold']].sort_values('sold',ascending=False).head(10)

In [ ]:
print(df[df['product_id'] == 'BC5']['sold'].gt(100).sum())
print((df[df['product_id'] == 'BC5']['sold'].gt(100).mean() * 100))

In [ ]:
df[(df['product_id'] == 'BC5') & (df['sold'] > 100)][
    ['date', 'sold', 'price', 'is_promotion']
].sort_values('date')

In [ ]:
df[(df['product_id'] == 'BC5') & (df['sold'] >= 1000)]['sold'].value_counts().sort_index()

### Product-level outlier review

High values are flagged for investigation rather than automatically removed. A large sale may be a genuine bulk order; validate against approved source context before changing it.

In [ ]:
print('sold quantiles:')
print(df['sold'].quantile([0.90, 0.95, 0.99, 0.995, 0.999]))

print('rows above 100:', (df['sold'] > 100).sum())
print('rows above 1000:', (df['sold'] > 1000).sum())
print('max sold:', df['sold'].max())

In [ ]:
product_eda = (
    df.groupby(['product_id', 'category'], observed=True)['sold']
      .agg(
          records='size',
          total_sold='sum',
          median_sold='median',
          max_sold='max',
          p99=lambda x: x.quantile(0.99),
          count_over_1000=lambda x: (x > 1000).sum()
      )
      .reset_index()
)

print(product_eda.sort_values('max_sold', ascending=False).head(20))

In [ ]:
product_p99 = (
    df.groupby('product_id', observed=True)['sold']
      .quantile(0.99)
      .rename('product_p99')
      .reset_index()
)

outlier_rows = df.merge(product_p99, on='product_id', how='left')
outlier_rows = outlier_rows[outlier_rows['sold'] > outlier_rows['product_p99']]

print(outlier_rows[
    ['product_id', 'category', 'date', 'sold', 'product_p99', 'price', 'is_promotion']
].sort_values('sold', ascending=False).head(20))

### Forecast origin and proxy definitions

`recency` is days since the previous nonzero sale. `last_sold` is the previous nonzero sale quantity, **not** inventory on hand. `lead_time` is a historical gap between sales events, **not** supplier lead time. Current inventory is not present in the input.

## 4. Causal time-series feature engineering

Lag and rolling features are calculated within product after sorting by date. Rolling statistics use a one-row-shifted sales series so they exclude the current day's sale. The historical sales-gap field is a demand-pattern proxy, **not supplier delivery lead time**.


In [ ]:
daily = daily.sort_values(['product_id', 'date']).reset_index(drop=True)

for lag in [1, 7, 14, 28, 56, 365]:
    daily['lag_' + str(lag)] = (
        daily.groupby('product_id', observed=True)['sold'].shift(lag).astype('float32')
    )


In [ ]:
previous_sold = daily.groupby('product_id', observed=True)['sold'].shift(1)

for window in [7, 14, 30, 60]:
    rolling = previous_sold.groupby(daily['product_id'], observed=True).rolling(
        window, min_periods=1
    )
    daily['mean_' + str(window)] = (
        rolling.mean().reset_index(level=0, drop=True).astype('float32')
    )


In [ ]:
previous_sold = daily.groupby('product_id', observed=True)['sold'].shift(1)

for window in [7, 30, 60]:
    rolling = previous_sold.groupby(daily['product_id'], observed=True).rolling(
        window, min_periods=1
    )
    daily['median_' + str(window)] = (
        rolling.median().reset_index(level=0, drop=True).astype('float32')
    )


In [ ]:
previous_sold = daily.groupby('product_id', observed=True)['sold'].shift(1)

for window in [7, 14, 30, 60]:
    rolling = previous_sold.groupby(daily['product_id'], observed=True).rolling(
        window, min_periods=1
    )
    daily['sum_' + str(window)] = (
        rolling.sum().reset_index(level=0, drop=True).astype('float32')
    )


In [ ]:
previous_sold = daily.groupby('product_id', observed=True)['sold'].shift(1)

rolling_7 = previous_sold.groupby(daily['product_id'], observed=True).rolling(7, min_periods=2)
daily['sales_std_7'] = rolling_7.std().reset_index(level=0, drop=True).astype('float32')

rolling_7 = previous_sold.groupby(daily['product_id'], observed=True).rolling(7, min_periods=1)
daily['max_7'] = rolling_7.max().reset_index(level=0, drop=True).astype('float32')

previous_nonzero = previous_sold.gt(0).astype('int8')
daily['nonzero_7'] = (
    previous_nonzero.groupby(daily['product_id'], observed=True)
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
    .astype('float32')
)

daily['nonzero_30'] = (
    previous_nonzero.groupby(daily['product_id'], observed=True)
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
    .astype('float32')
)


In [ ]:
# recency and lead time

daily = daily.sort_values(['product_id', 'date']).reset_index(drop=True)

last_sale_date = daily['date'].where(daily['sold'] > 0)
last_sale_qty = daily['sold'].where(daily['sold'] > 0)

last_sale_date = last_sale_date.groupby(daily['product_id'], observed=True).ffill()
last_sale_qty = last_sale_qty.groupby(daily['product_id'], observed=True).ffill()

previous_sale_date = last_sale_date.groupby(daily['product_id'], observed=True).shift(1)
previous_sale_qty = last_sale_qty.groupby(daily['product_id'], observed=True).shift(1)

daily['recency'] = (daily['date'] - previous_sale_date).dt.days.fillna(365).astype('float32')
daily['last_sold'] = previous_sale_qty.fillna(0).astype('float32')

sales_gap = (daily['date'].where(daily['sold'] > 0) - previous_sale_date).dt.days

daily['lead_time'] = sales_gap.groupby(daily['product_id'], observed=True).transform(
    lambda x: x.shift(1).expanding().median()
).fillna(1).astype('float32')

daily['month'] = daily['date'].dt.month.astype('int8')
daily['day_of_week'] = daily['date'].dt.dayofweek.astype('int8')


## 5. Seven-day target construction

For a product at date *t*, `future_7` is the sum of sales on *t+1* through *t+7*. It is a seven-day total, not a day-by-day forecast trajectory. The final forecast-origin date has no observed target yet.


In [ ]:
daily['future_7'] = (
    daily.groupby('product_id', observed=True)['sold']
         .transform(lambda x: x.rolling(7).sum().shift(-7))
).astype('float32')

daily = daily.sort_values(['date', 'product_id']).reset_index(drop=True)

print('future_7 distribution:')
print(daily['future_7'].describe(percentiles=[0.90, 0.95, 0.99, 0.995]))


In [ ]:
daily.head(20)

In [ ]:
print(daily.isna().sum())


In [ ]:
latest_date = daily['date'].max()
latest_full = daily[daily['date'] == latest_date].copy()
del latest_date


## 6. Modeling sample and chronological split

Rows missing a required lag, rolling feature, price, or target are removed from the training/evaluation sample. The latest-date rows are retained separately for inference. The split is chronological: the training cutoff is moved seven calendar days before the split date, and test origins are strictly after it, reducing overlap between seven-day labels.


In [ ]:
daily = daily.dropna(subset=[
    'future_7',
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'lag_56',
    'lag_365',
    'mean_7',
    'mean_14',
    'mean_30',
    'mean_60',
    'median_7',
    'median_30',
    'median_60',
    'sum_7',
    'sum_14',
    'sum_30',
    'sum_60',
    'sales_std_7',
    'max_7',
    'price'
])


In [ ]:
features = [
    'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_56', 'lag_365',
    'mean_7', 'mean_14', 'mean_30', 'mean_60',
    'median_7', 'median_30', 'median_60',
    'sum_7', 'sum_14', 'sum_30', 'sum_60',
    'sales_std_7', 'max_7', 'nonzero_7', 'nonzero_30',
    'recency', 'last_sold', 'lead_time',
    'price', 'is_promotion', 'month', 'day_of_week'
]

x = daily[features]

y = daily['future_7']


In [ ]:
print(x.shape)
print(y.shape)
print(x.isna().sum())
print(y.isna().sum())


In [ ]:
split_date = daily['date'].quantile(0.8)
train_end = split_date - pd.Timedelta(days=7)

print('train end:', train_end)
print('test starts after:', split_date)


### Global-model feature encoding

The global XGBoost model includes product, category, and `hcode` categoricals alongside numeric demand features.

## 7. Global XGBoost model

The global model uses numeric time-series features plus product, category, and `hcode` as native categoricals. Categories with fewer than 100 units sold in the training-period aggregate are routed to this model.


In [ ]:
from xgboost import XGBRegressor

category_columns = ['product_id', 'category', 'hcode']
global_features = features + category_columns

global_data = daily[global_features].copy()

for col in category_columns:
    global_data[col] = global_data[col].astype(str).replace('nan', 'Unknown').astype('category')

global_cat_levels = {
    col: global_data[col].cat.categories
    for col in category_columns
}

train_mask = daily['date'] <= train_end
x_train_global = global_data.loc[train_mask]
y_train_global = daily.loc[train_mask, 'future_7']

best_xgb = XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    tree_method='hist',
    enable_categorical=True,
    n_jobs=-1,
    random_state=42
)

best_xgb.fit(x_train_global, y_train_global)

del global_data, x_train_global, y_train_global, train_mask
print('global XGBoost model ready')


In [ ]:
print('categories:', daily['category'].nunique())


In [ ]:
category_sold = (
    df.groupby('category', observed=True)['sold']
      .sum()
)

category_sold_train = (
    df[df['date'] <= train_end]
      .groupby('category', observed=True)['sold']
      .sum()
)

### Category-level routing rules

Categories with fewer than 100 training-period units use the global XGBoost model. Other categories compare a baseline, ordinary CatBoost, and log-target CatBoost in a chronological validation slice; selected methods are applied per product. Sparse validation and constant-target cases have explicit fallbacks.

## 8. Category-level validation and model routing

For eligible categories, validation compares the 30-day baseline, raw-target CatBoost, and `log1p`-target CatBoost by product using RMSE. The selected learned model is refit on the full training period. Insufficient validation data falls back to the baseline; a constant training target uses a constant prediction.


In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

category_model = {}
category_log_model = {}
category_constant = {}
category_route_type = {}
product_method = {}
category_results = []
test_predictions = []

category_features = features + ['product_id', 'hcode']
validation_start = train_end - pd.Timedelta(days=120)
validation_train_end = validation_start - pd.Timedelta(days=7)

for category, category_data in daily.groupby('category', sort=False, observed=True):

    train = category_data[category_data['date'] <= train_end].copy()
    test = category_data[category_data['date'] > split_date].copy()
    sold_count = category_sold_train.get(category, 0)

    if sold_count < 100 or len(train) == 0:
        route = 'global'
    else:
        route = 'category'

    category_route_type[category] = route

    if len(test) == 0:
        continue

    y_test_category = test['future_7'].copy()
    baseline_test = (test['mean_30'].fillna(0) * 7).clip(lower=0)

    if route == 'global':

        x_test_global = test[global_features].copy()
        for col in category_columns:
            x_test_global[col] = pd.Categorical(
                x_test_global[col].astype('string').fillna('Unknown'),
                categories=global_cat_levels[col]
            )

        y_prediction = np.maximum(best_xgb.predict(x_test_global), 0)
        test_method = pd.Series('global', index=test.index)

        for product in test['product_id'].astype(str).unique():
            product_method[(category, product)] = 'global'

    else:

        x_train_category = train[category_features].copy()
        y_train_category = train['future_7'].copy()

        x_train_category['product_id'] = x_train_category['product_id'].astype(str)
        x_train_category['hcode'] = x_train_category['hcode'].astype('string').fillna('Unknown').astype(str)

        if y_train_category.nunique() == 1:

            value = y_train_category.iloc[0]
            category_constant[category] = value
            test_method = pd.Series('constant', index=test.index)

            for product in pd.concat([train['product_id'], test['product_id']]).astype(str).unique():
                product_method[(category, product)] = 'constant'

            y_prediction = np.full(len(test), value)

        else:

            validation_train = train[train['date'] <= validation_train_end].copy()
            validation_data = train[train['date'] > validation_start].copy()

            chosen_methods = {}

            # Validation mein baseline, normal CatBoost aur log CatBoost compare karna hai
            if (
                len(validation_train) > 100 and
                len(validation_data) > 20 and
                validation_train['future_7'].nunique() > 1
            ):

                x_val_train = validation_train[category_features].copy()
                y_val_train = validation_train['future_7'].copy()
                x_val = validation_data[category_features].copy()
                y_val = validation_data['future_7'].copy()

                for col in ['product_id', 'hcode']:
                    x_val_train[col] = x_val_train[col].astype('string').fillna('Unknown').astype(str)
                    x_val[col] = x_val[col].astype('string').fillna('Unknown').astype(str)

                baseline_val = (validation_data['mean_30'].fillna(0) * 7).clip(lower=0).to_numpy()

                val_model = CatBoostRegressor(
                    iterations=100,
                    depth=6,
                    learning_rate=0.05,
                    loss_function='RMSE',
                    verbose=False,
                    random_seed=42
                )
                val_model.fit(x_val_train, y_val_train, cat_features=['product_id', 'hcode'])
                pred_val = np.maximum(val_model.predict(x_val), 0)

                val_log_model = CatBoostRegressor(
                    iterations=100,
                    depth=6,
                    learning_rate=0.05,
                    loss_function='RMSE',
                    verbose=False,
                    random_seed=42
                )
                val_log_model.fit(
                    x_val_train,
                    np.log1p(y_val_train.clip(lower=0)),
                    cat_features=['product_id', 'hcode']
                )
                pred_val_log = np.maximum(np.expm1(val_log_model.predict(x_val)), 0)

                val_check = validation_data[['product_id', 'future_7']].copy()
                val_check['product_id'] = val_check['product_id'].astype(str)
                val_check['baseline'] = baseline_val
                val_check['category'] = pred_val
                val_check['category_log'] = pred_val_log

                for product in val_check['product_id'].unique():
                    product_rows = val_check[val_check['product_id'] == product]
                    actual = product_rows['future_7']

                    errors = {
                        'baseline': np.sqrt(mean_squared_error(actual, product_rows['baseline'])),
                        'category': np.sqrt(mean_squared_error(actual, product_rows['category'])),
                        'category_log': np.sqrt(mean_squared_error(actual, product_rows['category_log']))
                    }

                    chosen_methods[product] = min(errors, key=errors.get)

            # Validation data na ho toh baseline se start karna hai
            all_products = pd.concat([train['product_id'], test['product_id']]).astype(str).unique()
            for product in all_products:
                product_method[(category, product)] = chosen_methods.get(product, 'baseline')

            test_method = pd.Series(
                [product_method.get((category, str(product)), 'baseline') for product in test['product_id']],
                index=test.index
            )

            # Ab full training data par sirf required category models train karne hain
            methods_needed = set(test_method.unique())

            if 'category' in methods_needed:
                model = CatBoostRegressor(
                    iterations=150,
                    depth=6,
                    learning_rate=0.05,
                    loss_function='RMSE',
                    verbose=False,
                    random_seed=42
                )
                model.fit(x_train_category, y_train_category, cat_features=['product_id', 'hcode'])
                category_model[category] = model

            if 'category_log' in methods_needed:
                log_model = CatBoostRegressor(
                    iterations=150,
                    depth=6,
                    learning_rate=0.05,
                    loss_function='RMSE',
                    verbose=False,
                    random_seed=42
                )
                log_model.fit(
                    x_train_category,
                    np.log1p(y_train_category.clip(lower=0)),
                    cat_features=['product_id', 'hcode']
                )
                category_log_model[category] = log_model

            y_prediction = baseline_test.to_numpy().copy()

            model_mask = test_method == 'category'
            if model_mask.any() and category in category_model:
                x_test_cat = test.loc[model_mask, category_features].copy()
                x_test_cat['product_id'] = x_test_cat['product_id'].astype(str)
                x_test_cat['hcode'] = x_test_cat['hcode'].astype('string').fillna('Unknown').astype(str)
                y_prediction[model_mask.to_numpy()] = np.maximum(
                    category_model[category].predict(x_test_cat), 0
                )

            log_mask = test_method == 'category_log'
            if log_mask.any() and category in category_log_model:
                x_test_cat = test.loc[log_mask, category_features].copy()
                x_test_cat['product_id'] = x_test_cat['product_id'].astype(str)
                x_test_cat['hcode'] = x_test_cat['hcode'].astype('string').fillna('Unknown').astype(str)
                y_prediction[log_mask.to_numpy()] = np.maximum(
                    np.expm1(category_log_model[category].predict(x_test_cat)), 0
                )

            constant_mask = test_method == 'constant'
            if constant_mask.any():
                y_prediction[constant_mask.to_numpy()] = category_constant.get(category, 0)

    y_prediction = np.maximum(np.asarray(y_prediction), 0)

    test_result = test[['category', 'product_id', 'date', 'future_7']].copy()
    test_result['prediction'] = y_prediction
    test_result['baseline_prediction'] = baseline_test.to_numpy()
    test_result['model_used'] = test_method.to_numpy()
    test_predictions.append(test_result)

    mse = mean_squared_error(y_test_category, y_prediction)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_category, y_prediction) if y_test_category.nunique() > 1 else np.nan
    baseline_rmse = np.sqrt(mean_squared_error(y_test_category, baseline_test))

    category_results.append({
        'category': category,
        'route': route,
        'total_sold_train': sold_count,
        'model_type': test_method.mode().iloc[0] if len(test_method.mode()) else route,
        'mae': mean_absolute_error(y_test_category, y_prediction),
        'rmse': rmse,
        'mse': mse,
        'r2': r2,
        'baseline_mae': mean_absolute_error(y_test_category, baseline_test),
        'baseline_rmse': baseline_rmse,
        'model_better_than_baseline': rmse < baseline_rmse
    })

category_results = pd.DataFrame(category_results)

all_test_predictions = pd.concat(test_predictions, ignore_index=True)
product_results = []

for (category, product), product_data in all_test_predictions.groupby(['category', 'product_id'], observed=True):
    actual = product_data['future_7']
    predicted = product_data['prediction']
    mse = mean_squared_error(actual, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predicted) if actual.nunique() > 1 else np.nan

    product_results.append({
        'category': category,
        'product_id': product,
        'model_used': product_data['model_used'].iloc[0],
        'mae': mean_absolute_error(actual, predicted),
        'r2': r2,
        'rmse': rmse,
        'mse': mse,
        'test_rows': len(product_data)
    })

product_scores = pd.DataFrame(product_results)
product_list = df[['category', 'product_id']].drop_duplicates()
product_scores = product_list.merge(product_scores, on=['category', 'product_id'], how='left')
product_scores['model_used'] = product_scores['model_used'].fillna('not_scored')
product_scores = product_scores.sort_values('r2', ascending=True, na_position='last').reset_index(drop=True)


In [ ]:
test_actual = all_test_predictions["future_7"].to_numpy()
test_prediction = all_test_predictions["prediction"].to_numpy()
test_baseline = all_test_predictions["baseline_prediction"].to_numpy()
overall_test_metrics = pd.DataFrame([{
    "rows": len(all_test_predictions),
    "mae": mean_absolute_error(test_actual, test_prediction),
    "rmse": np.sqrt(mean_squared_error(test_actual, test_prediction)),
    "r2": r2_score(test_actual, test_prediction) if pd.Series(test_actual).nunique() > 1 else np.nan,
    "baseline_mae": mean_absolute_error(test_actual, test_baseline),
    "baseline_rmse": np.sqrt(mean_squared_error(test_actual, test_baseline)),
}])
display(overall_test_metrics)


## 9. Held-out evaluation summary

The following table is computed from the actual chronological test predictions. It is intentionally not stored in the committed notebook; clear outputs before committing. Metrics are data-dependent.

## 10. Latest-date forecast and preliminary replenishment

The notebook applies the selected route to the latest product rows and computes a provisional quantity as the seven-day forecast plus half the standard deviation of the preceding seven daily sales values, rounded upward. This is a heuristic: it does not subtract current or inbound inventory and is not calibrated to a service level.


In [ ]:
latest = latest_full.copy()

latest['model_method'] = [
    product_method.get((category, str(product)), 'baseline')
    for category, product in zip(latest['category'], latest['product_id'])
]

prediction_latest = np.zeros(len(latest))
category_features = features + ['product_id', 'hcode']

for category in latest['category'].unique():

    mask = latest['category'] == category
    rows = latest.loc[mask].copy()
    route = category_route_type.get(category, 'global')

    if route == 'global':

        x_latest = rows[global_features].copy()
        for col in category_columns:
            x_latest[col] = pd.Categorical(
                x_latest[col].astype('string').fillna('Unknown'),
                categories=global_cat_levels[col]
            )
        prediction = np.maximum(best_xgb.predict(x_latest), 0)

    else:

        methods = rows['model_method']
        prediction = (rows['mean_30'].fillna(0) * 7).clip(lower=0).to_numpy(copy=True)

        constant_mask = methods == 'constant'
        if constant_mask.any():
            prediction[constant_mask.to_numpy()] = category_constant.get(category, 0)

        raw_mask = methods == 'category'
        if raw_mask.any() and category in category_model:
            x_latest_cat = rows.loc[raw_mask, category_features].copy()
            x_latest_cat['product_id'] = x_latest_cat['product_id'].astype(str)
            x_latest_cat['hcode'] = x_latest_cat['hcode'].astype('string').fillna('Unknown').astype(str)
            prediction[raw_mask.to_numpy()] = np.maximum(
                category_model[category].predict(x_latest_cat), 0
            )

        log_mask = methods == 'category_log'
        if log_mask.any() and category in category_log_model:
            x_latest_cat = rows.loc[log_mask, category_features].copy()
            x_latest_cat['product_id'] = x_latest_cat['product_id'].astype(str)
            x_latest_cat['hcode'] = x_latest_cat['hcode'].astype('string').fillna('Unknown').astype(str)
            prediction[log_mask.to_numpy()] = np.maximum(
                np.expm1(category_log_model[category].predict(x_latest_cat)), 0
            )

    prediction_latest[mask.to_numpy()] = prediction

safety_stock_latest = 0.5 * latest['sales_std_7'].fillna(0).to_numpy()
reorder_quantity_latest = np.ceil(np.maximum(prediction_latest + safety_stock_latest, 0))

result_latest = latest[['category', 'product_id', 'date']].copy()
result_latest['predicted_7_day_sales'] = prediction_latest
result_latest['safety_stock'] = safety_stock_latest
result_latest['recommended_reorder_quantity'] = reorder_quantity_latest
result_latest = result_latest.sort_values('recommended_reorder_quantity', ascending=False)
result_latest.to_csv('latest_reorder_recommendations.csv', index=False)


## 11. Output files and interpretation

The notebook writes `latest_reorder_recommendations.csv` and `product_model_scores.csv` in the working directory. These files can contain product identifiers and data-derived results; do not commit them or publish them without explicit data-owner approval. The repository ignore rules exclude CSVs.


In [ ]:
global_categories_list = []
category_route_list = []

for category in sorted(df['category'].dropna().unique()):
    route = category_route_type.get(category, 'global')
    if route == 'global':
        global_categories_list.append(category)
    else:
        category_route_list.append(category)

print('categories using global XGBoost:')
print(global_categories_list)

print('\ncategories using category-side model:')
print(category_route_list)

product_scores.to_csv('product_model_scores.csv', index=False)

print('\nproduct-wise R2, RMSE and MSE (first 20 rows):')
from IPython.display import display

display(
    product_scores[
        ['category', 'product_id', 'model_used', 'mae', 'rmse', 'r2', 'mse']
    ].head(20)
)


## 12. Limitations and review checklist

- Missing product-date combinations are treated as zero sales; stockout/censored demand is not identified.
- Historical price and promotion features do not provide future-known promotion or price plans.
- The safety-stock calculation is a simple variability proxy, not an inventory policy.
- No current stock, inbound purchase orders, supplier delivery lead times, service-level targets, pack sizes, minimum order quantities, or shelf-life constraints are modeled.
- Evaluate errors and assumptions on an authorized dataset before using results in a planning process. No production or business-impact claims are made here.